In [1]:
import pandas as pd
import numpy as np
import os
import pickle
import json
# for content_based filtering
from sklearn.metrics.pairwise import cosine_similarity

# for collabaractive filtering

In [4]:
movieId_lookup = pd.read_csv('../dataset/processed/movieId_lookup.csv')
tmdb_movies_info = pd.read_csv('../dataset/processed/movies_df_clean.csv')



In [17]:
with open('../dataset/processed/userWatched_movies.json','r') as file:
    userWatched_movies = json.load(file)
userWatched_movies = pd.Series(userWatched_movies)

In [24]:
#  filter movies which have rated by users. 
tmdb_movies_info = tmdb_movies_info[tmdb_movies_info['movie_id'].isin(movieId_lookup['tmdb_movieId'])].reset_index(drop=True)

In [25]:
with open('../artifact/svd_model.pkl','rb') as file:
    svd_model = pickle.load(file)

In [26]:
combined_embeddings = np.load('../artifact/embeddings/final_embeddings.npy')
director_embeddings = np.load('../artifact/embeddings/director_embeddings.npy')
keyword_embeddings = np.load('../artifact/embeddings/keyword_embeddings.npy')
genre_embeddings = np.load('../artifact/embeddings/genre_embeddings.npy')
overview_embeddings = np.load('../artifact/embeddings/overview_embeddings.npy')


In [134]:

def content_based_filtering(movie_idx):
    '''
    apply content based filtering on given movie index and return the top similarities movies.
    input:
    movie_idx : index value of movieId_lookup table.

    return :
    top similarity movies list.
    '''
   
    # find the index of tmdb movie based on index value
    tmdb_movieId = movieId_lookup.iloc[movie_idx]['tmdb_movieId']
    
    # calculate movie similarity based on embedded text vector metrix.
    tmdb_index = tmdb_movies_info[tmdb_movies_info['movie_id'] == tmdb_movieId].index[0]
    similarities = cosine_similarity([combined_embeddings[tmdb_index]],combined_embeddings)[0]

    # sort index based on values then change the order into descending and give first 10 values index
    top_indicies = similarities.argsort()[::-1][:11]

    

    # for similary_tmdb_id in top_indicies:
    for movie_title in  tmdb_movies_info.loc[top_indicies,'title']:
        print(movie_title)
    # return tmdb_movies_info.loc[top_indicies,'title']
    

2925                                  Murderball
2152                   Michael Jordan to the Max
2985                                    Top Spin
2876                                    Wordplay
1274                              Without Limits
2852                                 Hoop Dreams
2548                               Riding Giants
2606                           Touching the Void
2904    Born to Fly: Elizabeth Streb vs. Gravity
2874                            Mad Hot Ballroom
2877                              Beyond the Mat
Name: title, dtype: str

In [17]:
PATH_MOVIELENS = os.path.join('..','dataset','MovieLens')

movielens_movies = pd.read_csv(os.path.join(PATH_MOVIELENS,'movies.csv'))


In [52]:
def collaborative_based_filtering(user_id):
    '''
    apply collaborative based filtering on given movie index and return the top similarities movies.
    input:
    movie_idx : index value of movieId_lookup table.

    return :
    top similarity movies list.
    ''' 

     # find the index of movielens movie based on index value
    # get all movies which exist in dataset.
    all_movies = set(movieId_lookup['movieLens_movieId'])
    
    movies_watched = set(userWatched_movies[str(user_id)])
    
    if not type(movies_watched) == set:
        movie_watched = set(movie_watched)
        
    unwatched_movies = all_movies - movies_watched

    
    predictions = []
    # calculate the prediction rating for user U for unwatched movie uM
    for movie in unwatched_movies:
        pred_rating = svd_model.predict(user_id,movie).est
        predictions.append([movie,pred_rating])
       
    
    # sort the predicted rating by highly rated movie prediction
    predictions.sort(key=lambda x : x[1],reverse=True)


    
    movie_lookup = dict(zip(movieId_lookup['movieLens_movieId'],movieId_lookup['title_clean']))
    
    for movieId, rating in predictions[:11]:
        print(movie_lookup[movieId],' : ',rating)
    

    
    

In [53]:
collaborative_based_filtering(519)

Harry potter and the half-blood prince  :  4.522764175597403
Fear and loathing in las vegas  :  4.2696316058748645
Gran torino  :  4.262284979252038
Hotel rwanda  :  4.242269200661832
Lost in translation  :  4.219929579008488
Blood diamond  :  4.138431951574938
Welcome to the dollhouse  :  4.129870837309026
Cinderella man  :  4.122785159070314
American history x  :  4.094437673941751
Mission: impossible - ghost protocol  :  4.083773602015143
Children of men  :  4.0837487649619435


In [ ]:
content_based_filtering(movieId_lookup.sample(1).index[0])

In [131]:
movie_id = movieId_lookup.sample(1).index[0]
user_id = '108795'
print(f' movie_id {movie_id}')
print(f' user_id {user_id}')

 movie_id 1279
 user_id 108795


In [135]:
content_based_filtering(movie_id)

Ironclad
Kingdom of Heaven
Henry V
First Knight
In the Name of the King: A Dungeon Siege Tale
Pirates of the Caribbean: On Stranger Tides
Elizabeth: The Golden Age
Hell's Angels
The Last of the Mohicans
Barry Lyndon
Ladyhawke


In [133]:
collaborative_based_filtering(user_id)

One flew over the cuckoo's nest  :  4.253267246770918
Apocalypse now  :  4.235958151441624
It happened one night  :  4.226346622163668
Dr. strangelove or: how i learned to stop worrying and love the bomb  :  4.197599273246513
Some like it hot  :  4.172173785547581
Rebecca  :  4.166519797483818
Midnight cowboy  :  4.164186285028516
Henry v  :  4.162900041991528
Monty python and the holy grail  :  4.156676832042193
Taxi to the dark side  :  4.154126559409125
Lawrence of arabia  :  4.143300164788154


In [130]:
userWatched_movies.index

Index(['519', '4019', '5807', '6862', '13803', '24236', '29179', '30178',
       '37824', '40491', '43091', '44772', '46546', '47156', '48541', '49727',
       '51278', '52020', '57743', '57826', '60461', '61941', '66926', '69502',
       '71185', '75154', '79045', '87418', '88392', '91338', '96408', '102957',
       '107409', '108795', '109459', '111724', '111770', '113946', '114613',
       '119499', '120525', '129396', '130379', '144475', '147756', '148441',
       '149753', '156131', '158866', '161939'],
      dtype='str')